In [2]:
import pandas as pd
import re
from collections import Counter
import json
import numpy as np

# 1. Load Datasets
df_shapes = pd.read_csv('sentence_dataset_shapes.csv')
df_numbers = pd.read_csv('sentence_dataset_numbers.csv')
df_tictactoe = pd.read_csv('sentence_dataset_tictactoe.csv')

# Combine all text for global vocabulary analysis
all_sentences = pd.concat([df_shapes['sentence'], df_numbers['sentence'], df_tictactoe['sentence']])

def custom_tokenizer(text):
    """
    Standardizes text and preserves hyphenated spatial terms (e.g., 'top-left').
    """
    text = text.lower()
    # Regex to capture words and hyphenated position terms as single tokens
    tokens = re.findall(r'\b\w+(?:-\w+)*\b', text)
    return tokens

# 2. Build Vocabulary
all_tokens = []
for s in all_sentences:
    all_tokens.extend(custom_tokenizer(s))

vocab_counts = Counter(all_tokens)
# Mapping: 0 for Padding, 1 for Unknown tokens
vocab = {word: i+2 for i, (word, _) in enumerate(vocab_counts.items())}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"Total Vocabulary Size: {len(vocab)}")
print("Top 10 tokens in vocab:", list(vocab.keys())[:10])

# 3. Export Vocabulary for team consistency
with open('vocab_mapping.json', 'w') as f:
    json.dump(vocab, f)

# Example: Convert sentence to fixed-length ID sequences
def sentence_to_ids(text, vocab, max_len=25):
    tokens = custom_tokenizer(text)
    ids = [vocab.get(t, 1) for t in tokens]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return ids[:max_len]

# Apply preprocessing
df_shapes['tokenized_ids'] = df_shapes['sentence'].apply(lambda x: sentence_to_ids(x, vocab))
print("Example of preprocessed sequence:", df_shapes['tokenized_ids'].iloc[0])

Total Vocabulary Size: 1735
Top 10 tokens in vocab: ['a', 'large', 'square', 'is', 'above', 'small', 'triangle', 'there', 'pink', 'hexagon']
Example of preprocessed sequence: [2, 3, 4, 5, 6, 2, 7, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
